# Hallucination Detection with Faithfulness & Groundedness

Score RAG outputs for faithfulness and groundedness to catch hallucinations before they reach users.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/quickstart-notebooks/quickstart/hallucination-detection.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on_GitHub-181717?logo=github&logoColor=white)](https://github.com/future-agi/cookbooks/blob/cookbook/quickstart-notebooks/quickstart/hallucination-detection.ipynb)


By the end of this guide you will have scored an LLM response for faithfulness, scored it for groundedness, and combined both metrics in a single `evaluate()` call.

**Prerequisites**
- FutureAGI account - [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` (see [Get your API keys](https://docs.futureagi.com/admin-settings))
- Python 3.9+

## Install

The `[nli]` extra installs the local NLI model used by `faithfulness` and `contradiction_detection`. Without it, these metrics fall back to a less accurate word-overlap heuristic.

In [ ]:
%pip install 'ai-evaluation[nli]' --quiet

## Set API keys

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-api-key"
os.environ["FI_SECRET_KEY"] = "your-secret-key"

## Metrics for hallucination detection

Three built-in metrics cover hallucination detection. Local NLI metrics run on your machine with no API key; Turing metrics use FutureAGI's purpose-built evaluation models.

| Metric | Engine | Required inputs | Output | What it catches |
|---|---|---|---|---|
| `faithfulness` | Local NLI | `output, context` | score 0-1 | Contradictions between output and context |
| `groundedness` | Turing or local | `output, input, context` | Pass/Fail | Output claims not traceable to context |
| `context_adherence` | Turing | `output, context` | score 0-1 | How strictly output stays within context boundaries |

## Step 1: Score faithfulness on a single response

This step checks whether the LLM response is consistent with the retrieved context -- no contradictions allowed.

In [ ]:
from fi.evals import evaluate

context = (
    "The James Webb Space Telescope (JWST) was launched on December 25, 2021. "
    "It orbits the Sun at the second Lagrange point (L2), approximately 1.5 million "
    "kilometers from Earth. JWST observes primarily in the infrared spectrum."
)

question = "When was the James Webb Space Telescope launched and where does it orbit?"

# A response that faithfully reflects the context
response = (
    "The James Webb Space Telescope was launched on December 25, 2021. "
    "It orbits the Sun at the L2 Lagrange point, about 1.5 million kilometers from Earth."
)

result = evaluate(
    "faithfulness",
    output=response,
    context=context,
    input=question,
)

print(f"Faithfulness score : {result.score:.2f}")
print(f"Passed             : {result.passed}")
print(f"Reason             : {result.reason}")

Now test a hallucinated response:

In [ ]:
from fi.evals import evaluate

context = (
    "The James Webb Space Telescope (JWST) was launched on December 25, 2021. "
    "It orbits the Sun at the second Lagrange point (L2), approximately 1.5 million "
    "kilometers from Earth. JWST observes primarily in the infrared spectrum."
)

question = "When was the James Webb Space Telescope launched and where does it orbit?"

hallucinated_response = (
    "The James Webb Space Telescope was launched on March 10, 2022. "
    "It orbits Earth at an altitude of 600 kilometers."
)

result = evaluate(
    "faithfulness",
    output=hallucinated_response,
    context=context,
    input=question,
)

print(f"Faithfulness score : {result.score:.2f}")
print(f"Passed             : {result.passed}")
print(f"Reason             : {result.reason}")

## Step 2: Check groundedness with the Turing engine

`groundedness` checks whether every claim in the output is traceable to the provided context. Unlike faithfulness (which flags direct contradictions), groundedness also catches plausible-sounding additions the model makes that have no basis in the context.

Test a response that adds unsourced facts:

In [ ]:
from fi.evals import evaluate

context = (
    "The James Webb Space Telescope (JWST) was launched on December 25, 2021. "
    "It orbits the Sun at the second Lagrange point (L2), approximately 1.5 million "
    "kilometers from Earth. JWST observes primarily in the infrared spectrum."
)

question = "When was the James Webb Space Telescope launched and where does it orbit?"

# A response that adds facts not present in the context
ungrounded_response = (
    "The James Webb Space Telescope was launched on December 25, 2021. "
    "It orbits the Sun at L2, 1.5 million kilometers from Earth. "
    "It is serviced every year by astronauts in low Earth orbit."
)

result = evaluate(
    "groundedness",
    output=ungrounded_response,
    context=context,
    input=question,
    model="turing_small",
)

print(f"Passed : {result.passed}")
print(f"Reason : {result.reason}")

Now test a clean response that stays within the context:

In [ ]:
clean_response = (
    "The James Webb Space Telescope was launched on December 25, 2021 "
    "and orbits the Sun at L2, about 1.5 million kilometers from Earth."
)

result = evaluate(
    "groundedness",
    output=clean_response,
    context=context,
    input=question,
    model="turing_small",
)

print(f"Passed : {result.passed}")
print(f"Reason : {result.reason}")

## Step 3: Combine both metrics in one evaluate() call

Pass a list of metric names to run faithfulness and groundedness together on a single output. Returns a `BatchResult` you can iterate or index by name.

In [ ]:
from fi.evals import evaluate

context = (
    "The Great Barrier Reef is the world's largest coral reef system, located in the "
    "Coral Sea off the coast of Queensland, Australia. It is composed of over 2,900 "
    "individual reefs and 900 islands stretching over 2,300 kilometers."
)

question = "Where is the Great Barrier Reef and how large is it?"

response = (
    "The Great Barrier Reef is located in the Coral Sea off Queensland, Australia. "
    "It spans over 2,300 kilometers and consists of more than 2,900 individual reefs "
    "and 900 islands."
)

results = evaluate(
    ["faithfulness", "groundedness"],
    output=response,
    context=context,
    input=question,
)

# Iterate over both results
for result in results:
    status = "PASS" if result.passed else "FAIL"
    print(f"{result.eval_name:<15} score={result.score:.2f}  {status}")
    print(f"  Reason: {result.reason}")
    print()

# Or look up by name directly
faith_result = results.get("faithfulness")
ground_result = results.get("groundedness")

print(f"Both metrics passed: {faith_result.passed and ground_result.passed}")

> **Tip:** `faithfulness` runs entirely locally via NLI -- no API key required. `groundedness` can also run locally (omit the `model=` argument) or via Turing models (e.g. `model="turing_small"`). `turing_small` balances speed and accuracy; use `turing_flash` for lowest latency or `turing_large` for highest accuracy. For high-volume pipelines, consider running evaluations concurrently with `concurrent.futures.ThreadPoolExecutor`.

## What you built

- Scored a RAG response for **faithfulness** (local NLI) to detect contradictions against the retrieved context -- no API key needed
- Used **groundedness** (Turing, Pass/Fail) to catch unsourced claims the LLM adds beyond the context
- Combined multiple metrics in a single `evaluate([...])` call returning a `BatchResult` with per-metric `.score`, `.passed`, and `.reason`

**Next steps:**
- [Running Your First Eval](https://docs.futureagi.com/cookbook/quickstart/first-eval) -- A broader introduction to local metrics, Turing models, and LLM-as-Judge in one guide.
- [Custom Eval Metrics](https://docs.futureagi.com/cookbook/quickstart/custom-eval-metrics) -- Define and register your own reusable evaluation rubric via the dashboard or SDK.
- [Eval in CI/CD](https://docs.futureagi.com/cookbook/quickstart/cicd-eval-pipeline) -- Block hallucinating prompts from shipping by wiring faithfulness checks into GitHub Actions.
- [All Built-in Metrics](https://docs.futureagi.com/future-agi/get-started/evaluation/builtin-evals/overview) -- Full reference for every built-in eval metric.